# DriveMind ML track -- base measurement, QLoRA fine-tune, fine-tuned measurement

Three numbers, one environment, in that order. A base figure from one
runtime and a tuned figure from another are not comparable, so both are
produced here, on the same pair of cards, in the same session.

**Before running:** Settings -> Accelerator = `GPU T4 x2`, and Internet =
`On` (needed for pip and the model download). Session cap is 12 h; the
whole sequence is an estimated 2.5-3.5 h, unmeasured.

**Both cards are used for every GPU-bound step**, but not the same way,
because training and evaluation parallelise differently:

*   **Training is data-parallel.** `torchrun --nproc_per_node=2` puts a
    full replica on each card, gives each a disjoint half of every batch,
    and all-reduces the LoRA gradients before the optimizer step.
    Deliberately *not* `device_map="auto"` across both cards, which is
    model parallelism: it splits one replica's layers over the two
    devices and runs them in sequence, so card 1 waits while card 0
    computes and the transfers make it slower than one card. A 4B model
    in 4-bit NF4 is about 2.5 GB and fits a 16 GB T4 many times over, so
    there is nothing to split.
*   **Evaluation is generation-sharded.** `ml.eval_sharded` runs one
    worker per card, each generating an interleaved share of the held-out
    cases, and then scores **all** of them with a single
    `evaluate_emitter` call in the parent process. Generation is the
    hour; scoring is a fraction of a second.

The per-rank batch and the accumulation steps are derived from
`EFFECTIVE_BATCH` in cell 1, so the optimizer sees the same 16 sequences
per step whether this runs on one card or two. A two-card run is the same
fine-tune finished sooner, not a different one at a larger batch.

**Nothing here scores anything.** Every metric comes from
`ml.eval_holdout` or `ml.eval_sharded`, and both get it from
`backend.services.ai_holdout`. This file is a sequencer with a stopwatch
-- a notebook that recomputed a metric would be a second scorer that can
disagree with the first.

**The order is the point.** Each step is cheaper than the next and can
refuse, so a stale corpus costs three minutes instead of three hours:

| Step | Cost | Cards | What it refuses on |
|---|---|---|---|
| deps | ~2 min | - | pip fails rather than replacing Kaggle's torch |
| corpus | ~1 min | - | `build_dataset` will not write a bad corpus |
| reference | ~10 s | - | split verification, on Linux, no GPU |
| shard self-test | ~10 s | - | sharded scoring differing from unsharded |
| NCCL preflight | ~1 min | 2 | the collective DDP needs, hanging |
| dry run | ~2 min | - | stale policy stamp, over-length row |
| smoke | ~15 min | 1 | model does not load, or emits no JSON |
| calibration | ~10 min | 2 | the per-rank batch not fitting |
| base eval | ~25 min | 2 | held-out rows are not the published ones |
| train | ~1-2 h | 2 | |
| tuned eval | ~25 min | 2 | |

The single-card version of this sequence was estimated at 4-6 h. Both
figures are estimates; neither has been measured.

**The smoke cell runs on one card on purpose**, and it has to run before
any two-process cell: it is where the ~8 GB of base weights are
downloaded, and two processes racing to populate one Hugging Face cache
is a slow way to learn about file locking.

When it finishes, paste `artifacts/base_holdout.txt` and
`artifacts/tuned_holdout.txt` back into the session that wrote this.

In [ ]:
# --- Cell 1: what this is running on, and the recipe --------------------
#
# Reported, not logged, because a held-out figure is only comparable with
# another if the runtime is known. 4-bit on one card and bf16 on another
# are different measurements of the same weights.

import os
import subprocess
import sys
import tempfile
from pathlib import Path

MODEL = "Qwen/Qwen3-4B"

# The recipe, held fixed across however many cards turn up. Cells 11 and
# 13 run one rank per card with PER_DEVICE_BATCH sequences each and
# GRAD_ACCUM accumulation steps, so the optimizer sees EFFECTIVE_BATCH
# sequences per step either way.
#
# If the calibration cell OOMs, set PER_DEVICE_BATCH = 2 and re-run it:
# that is the per-rank footprint train_qlora already defaults to,
# GRAD_ACCUM follows automatically, EFFECTIVE_BATCH does not move, and the
# only thing lost is throughput. Larger micro-batches are the reason to
# try 4 first -- fewer, bigger kernels on a card that has the memory.
EFFECTIVE_BATCH = 16
PER_DEVICE_BATCH = 4

print(f"python          {sys.version.split()[0]}")

assert sys.version_info >= (3, 10), (
    "backend/ uses zip(strict=True), so 3.10 is the hard floor. "
    "pyproject declares >=3.12, which is the tested floor, not this one."
)

try:
    import torch

except ImportError:
    raise SystemExit("torch is not installed -- is the accelerator set to GPU?")

print(f"torch           {torch.__version__}")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device visible. Settings -> Accelerator -> GPU T4 x2."
    )

card_names = []

for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    card_names.append(properties.name)

    print(
        f"device {index}        {properties.name}, "
        f"{properties.total_memory // (1024 * 1024)} MiB, "
        f"capability {properties.major}.{properties.minor}, "
        f"CUDA {torch.version.cuda}"
    )

print(f"bf16 supported  {torch.cuda.is_bf16_supported()}")

GPUS = [str(index) for index in range(torch.cuda.device_count())]
DEVICES = ",".join(GPUS)
WORLD_SIZE = len(GPUS)

GRAD_ACCUM, remainder = divmod(EFFECTIVE_BATCH, PER_DEVICE_BATCH * WORLD_SIZE)

if remainder or GRAD_ACCUM < 1:
    raise SystemExit(
        f"EFFECTIVE_BATCH={EFFECTIVE_BATCH} is not PER_DEVICE_BATCH="
        f"{PER_DEVICE_BATCH} x GRAD_ACCUM x WORLD_SIZE={WORLD_SIZE} for any "
        "whole GRAD_ACCUM >= 1. Pick a per-rank batch that divides it -- "
        "silently rounding would report a recipe that was not run."
    )

print()
print(f"  cards           {WORLD_SIZE} ({DEVICES})")
print(
    f"  recipe          {EFFECTIVE_BATCH} per step = {PER_DEVICE_BATCH} "
    f"per rank x {GRAD_ACCUM} accum x {WORLD_SIZE} rank(s)"
)

if WORLD_SIZE < 2:
    print()
    print("  ONE CARD VISIBLE. Everything below still runs -- torchrun with")
    print("  one rank is a single process that happens to have RANK set, and")
    print("  eval_sharded with one shard is eval_holdout with an extra corpus")
    print("  regeneration -- but expect roughly double the wall clock in the")
    print("  table above. Settings -> Accelerator -> GPU T4 x2.")

elif len(set(card_names)) > 1:
    print()
    print("  MIXED CARDS: " + ", ".join(sorted(set(card_names))))
    print("  Generation is sharded evenly and DDP steps in lockstep, so the")
    print("  slower card sets the wall clock for both.")

print()
print("  T4 is Turing (7.5) and has no bfloat16. pick_dtype() selects")
print("  fp16 there; passing bf16=True on a T4 raises rather than")
print("  falling back, which is why that is decided at runtime.")

In [ ]:
def sh(*command, cwd=None, env=None, check=True, timeout=None):
    """
    Run a command, stream its output into the notebook, return its code.

    No output capture: these steps run for tens of minutes and a
    progress bar you cannot see is indistinguishable from a hang.

    `timeout` is for the NCCL preflight and nothing else. A collective
    that is misconfigured does not raise, it waits, and on a
    session-limited runtime an unbounded wait is the expensive failure.
    """

    print(f"\n$ {' '.join(str(part) for part in command)}\n", flush=True)

    try:
        completed = subprocess.run(
            list(command), cwd=cwd, env=env, timeout=timeout
        )

    except subprocess.TimeoutExpired:
        raise SystemExit(
            f"no exit after {timeout}s -- killed. If that was the NCCL "
            "preflight, the collective hung, and the fine-tune would have "
            "hung the same way for hours. Restart the kernel to clear any "
            "worker processes that outlived it."
        )

    if check and completed.returncode != 0:
        raise SystemExit(
            f"exit {completed.returncode} -- stopping here rather than "
            "running the next step on a failed one"
        )

    return completed.returncode


def torchrun(*args, module=True, **kwargs):
    """
    Launch under `torch.distributed.run`, one rank per visible card.

    `--standalone` means the launcher picks its own rendezvous port, so a
    re-run after a killed launch does not collide with the socket the
    last one left behind -- which on a notebook is otherwise a confusing
    "address already in use" on the cell you just fixed.

    With WORLD_SIZE == 1 this is an ordinary single-process run that
    happens to have RANK and LOCAL_RANK set, which is exactly what
    `ml.train_qlora` reads to decide its device map.
    """

    launcher = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        f"--nproc_per_node={WORLD_SIZE}",
    ]

    if module:
        launcher.append("-m")

    return sh(*launcher, *[str(part) for part in args], **kwargs)

In [ ]:
# --- Cell 3: find the repository ----------------------------------------
#
# Two ways in, checked in this order:
#
#   1. This file's own checkout, when run as `python ml/kaggle_qlora.py`
#      on a machine that already has the repo. `__file__` is undefined in
#      a notebook cell, so on Kaggle this candidate simply does not exist.
#   2. A Kaggle Dataset containing the repo (Add Input -> Datasets). No
#      push, no token, no credentials. This is the path to use while the
#      ML track is unpushed.
#   3. `git clone REPO_URL`, for when it is pushed and public.
#
# There is deliberately no fourth way involving a personal access token.
# A notebook is a shared artifact and a token pasted into one is a
# committed secret.

REPO_URL = "https://github.com/Xtremephenom/DriveMind.git"

# Every file the sequence actually needs. Checked up front because the
# useful failure is "the ML track is not in this checkout", not
# `ModuleNotFoundError: ml` forty minutes later.
REQUIRED = (
    "backend/services/dataset/build.py",
    "backend/services/ai_holdout.py",
    "ml/hf_runner.py",
    "ml/eval_holdout.py",
    "ml/eval_sharded.py",
    "ml/train_qlora.py",
)

ON_KAGGLE = Path("/kaggle/working").exists()
WORKDIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()


def is_checkout(path: Path) -> bool:
    return (path / "backend" / "services" / "dataset" / "build.py").exists()


def find_repo() -> Path:
    local = []

    if "__file__" in globals():
        local.append(Path(__file__).resolve().parent.parent)

    local.append(Path.cwd())

    for candidate in local:
        if is_checkout(candidate):
            print(f"  using the checkout at {candidate}")
            return candidate

    # /kaggle/input/<dataset>/ or /kaggle/input/<dataset>/<subdir>/,
    # depending on whether the zip had a top-level folder.
    uploaded = sorted(Path("/kaggle/input").glob("*/")) + sorted(
        Path("/kaggle/input").glob("*/*/")
    )

    for candidate in uploaded:
        if not is_checkout(candidate):
            continue

        print(f"  found an uploaded checkout at {candidate}")

        # /kaggle/input is read-only and the corpus build writes
        # data/*.jsonl, so it is copied out rather than used in place.
        # An existing copy is reused; delete WORKDIR/DriveMind by hand
        # after swapping the attached dataset for a newer one.
        import shutil

        destination = WORKDIR / "DriveMind"

        if not destination.exists():
            shutil.copytree(candidate, destination)
            print(f"  copied to {destination} (input is read-only)")

        return destination

    destination = WORKDIR / "DriveMind"

    if not is_checkout(destination):
        print("  no local or uploaded checkout found; cloning")
        sh("git", "clone", "--depth", "1", REPO_URL, str(destination))

    return destination


REPO = find_repo()

missing = [name for name in REQUIRED if not (REPO / name).exists()]

if missing:
    raise SystemExit(
        "This checkout is incomplete:\n  "
        + "\n  ".join(missing)
        + "\n\nThe ML track is not in origin/main yet. Either push it, or "
        "zip the working tree, upload it as a Kaggle Dataset, and attach "
        "it with Add Input."
    )

os.chdir(REPO)
sys.path.insert(0, str(REPO))

ARTIFACTS = REPO / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

print(f"\n  repo            {REPO}")
print(f"  artifacts       {ARTIFACTS}")

if (REPO / ".git").exists():
    sh("git", "log", "-1", "--format=  commit          %h %s", cwd=REPO)

else:
    print("  commit          unknown (uploaded as files, not a checkout)")

In [ ]:
# --- Cell 4: dependencies, with torch protected -------------------------
#
# Kaggle's torch is paired with the image's CUDA runtime and driver.
# Letting pip resolve a fresh one is a multi-gigabyte download that can
# also break that pairing, and it does it quietly -- the install
# succeeds and the failure shows up later as a CUDA error.
#
# So torch goes into a constraints file pinned to the *exact* installed
# version, local label included (`2.9.0+cu126`). If the other four can
# be satisfied against it, they install and torch is untouched. If they
# cannot, pip cannot find that version on PyPI either and fails loudly.
# A loud failure here is the correct outcome: it means the pin set and
# this image disagree, which is a decision for a person.

import importlib.metadata

PINS = (
    "transformers==5.16.1",
    "peft==0.20.0",
    "accelerate==1.14.0",
    "bitsandbytes==0.50.2",
)

torch_before = torch.__version__

constraints = Path(tempfile.gettempdir()) / "drivemind-constraints.txt"
constraints.write_text(f"torch=={torch_before}\n", encoding="utf-8")

print(f"  holding         torch=={torch_before}")

sh(
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--constraint",
    str(constraints),
    *PINS,
)

print()

for name in ("torch", "transformers", "peft", "accelerate", "bitsandbytes"):
    try:
        print(f"  {name:<16}{importlib.metadata.version(name)}")

    except importlib.metadata.PackageNotFoundError:
        print(f"  {name:<16}NOT INSTALLED")

torch_after = importlib.metadata.version("torch")

if torch_after != torch_before:
    raise SystemExit(
        f"torch changed from {torch_before} to {torch_after} despite the "
        "constraint. Stop: the CUDA pairing is no longer the image's, and "
        "any number measured now is measured on an unknown runtime."
    )

print("\n  torch unchanged. The pins were resolved against CPython 3.14.2")
print("  win_amd64; if a linux wheel for one of them is missing, the pip")
print("  step above is where you find out.")

In [ ]:
# --- Cell 5: rebuild the corpus, and check it against Windows -----------
#
# `data/` is gitignored, so there are no rows here until this runs. The
# build is deterministic from seed 42 and refuses to write on duplicate
# case_id, cross-file overlap, a missing FileCategory, a signal outside
# the production vocabulary, or a prompt containing the answer.
#
# The hashes below were measured on the Windows machine that built the
# corpus the dataset card describes. Comparing against them turns "the
# corpus should be reproducible on Linux" into a measurement.
#
# They are **LF-normalized**. `writer.write_jsonl` opens with the default
# newline translation, so the same rows are CRLF on Windows and LF on
# Linux: raw bytes differ for a reason that has nothing to do with the
# corpus. Normalizing isolates a real content difference from a line
# ending, which is the only comparison worth making across platforms.
#
# A mismatch does not invalidate the run -- `evaluate_emitter`
# regenerates and verifies in place, so the scoring is self-consistent
# either way. It would mean the distributions in the dataset card were
# measured on a different corpus than the one being scored, which is
# worth knowing before a number is published.

import hashlib

WINDOWS_SHA256_LF = {
    "train": "103ea0face08040947f053b1aecd326b2d8baed1369eb7334d2ab612dc867630",
    "validation": "8f051377ee146db22060015a27bd388bfa4b40308d3b3bfbdf1ae0ac13b4e739",
    "test": "3ddb91f5a95b6fbf63d765ccf9b8f1e7153a599176333a00389ce379c41366ca",
    "gold": "2882da3182d2f480ea1d65cdb6f22f7d9711f6b37b1fc209b144f2d81a125d70",
    "red_team": "215b455196ef40b03934af70488d144236242a7b5b64547487b9d700f3693e3e",
}

from backend.services.dataset.build import build_dataset
from backend.services.decision.engine import POLICY_VERSION

print(f"  policy version  {POLICY_VERSION}\n")

summary = build_dataset()

print()

divergent = []

for name, expected in WINDOWS_SHA256_LF.items():
    raw = (REPO / "data" / f"{name}.jsonl").read_bytes()
    normalized = raw.replace(b"\r\n", b"\n")
    actual = hashlib.sha256(normalized).hexdigest()

    rows = len(normalized.splitlines())
    verdict = "match" if actual == expected else "DIVERGES from Windows"

    if actual != expected:
        divergent.append(name)

    print(f"  {name:<12}{rows:>6} rows  {actual[:16]}  {verdict}")

if divergent:
    print(
        "\n  WARNING: "
        + ", ".join(divergent)
        + " differ in content from the Windows build.\n"
        "  The run below is still internally consistent, but the dataset\n"
        "  card's measured distributions describe the other corpus."
    )

else:
    print("\n  All five match the Windows build, content-for-content.")

In [ ]:
# --- Cell 6: the two environments, and the deterministic reference -------
#
# `RuleBasedAIProvider` is a mirror of the engine, so this is 100/100/100
# with 0 unsafe escalations by construction. It is run anyway, and first,
# for two reasons: it proves the held-out verification works on this
# machine before any GPU time is spent, and it is the reference row the
# model's numbers get read against. A model figure with nothing beside it
# is hard to interpret; the same figure next to a known ceiling is not.
#
# Takes seconds, on the CPU. If it fails, nothing below is worth starting.
#
# Two environments from here on, and which one a cell gets is a decision:
#
#   RUN_ENV       every card visible. For `torchrun`, which assigns one
#                 rank per card, and for `ml.eval_sharded`, which spawns
#                 one worker per card and sets CUDA_VISIBLE_DEVICES for
#                 each of them itself.
#   ONE_CARD_ENV  the first card only. For the smoke run, which is a
#                 single load and also the ~8 GB download.
#
# The NCCL pair is insurance, not tuning. Kaggle's two T4s have no
# NVLink, so peer-to-peer already falls back through the host; saying so
# explicitly costs nothing at this scale -- LoRA gradients are tens of
# megabytes a few hundred times -- and it removes the most common way a
# two-GPU run hangs instead of failing. A hang is the expensive failure
# on a session-limited runtime.

RUN_ENV = {
    **os.environ,
    "PYTHONPATH": str(REPO),
    # A tokenizers fork warning on every one of 1,126 generations is
    # noise that hides the real output.
    "TOKENIZERS_PARALLELISM": "false",
    "NCCL_P2P_DISABLE": "1",
    "NCCL_IB_DISABLE": "1",
    # Four vCPUs shared by WORLD_SIZE ranks. Left alone, each rank's
    # OpenMP pool sizes itself to all of them and they contend.
    "OMP_NUM_THREADS": str(
        max(1, (os.cpu_count() or 2) // max(WORLD_SIZE, 1))
    ),
}

ONE_CARD_ENV = {**RUN_ENV, "CUDA_VISIBLE_DEVICES": GPUS[0]}

print(
    f"  OMP_NUM_THREADS {RUN_ENV['OMP_NUM_THREADS']} per rank "
    f"({os.cpu_count()} vCPU, {WORLD_SIZE} rank(s))"
)

sh(
    sys.executable,
    "-m",
    "backend.services.ai_holdout",
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 7: is the sharded evaluator the same evaluator? ----------------
#
# The two held-out cells below do not call `ml.eval_holdout`. They call
# `ml.eval_sharded`, which generates on both cards in two processes and
# then scores the merged output in one. That is a second path to a
# published number, and a second path is the kind of thing that quietly
# disagrees with the first.
#
# So it is checked here, on the CPU, before any of it matters. The
# deterministic provider is run through the sharded two-pass path at 1, 2,
# 3 and 4 shards -- none of which divides 1,126 evenly -- and through a
# plain single-process `evaluate_emitter`, and the reports have to come
# out byte-identical. Then six refusals are exercised rather than
# asserted: a case no shard generated, a case two shards both generated, a
# response matching no case, shards that ran different models, a missing
# shard, and files from two different runs.
#
# Ten seconds, no GPU, no model, no network. If this fails, the numbers
# below would come from an evaluator that does not agree with the one the
# repository documents, and the right move is `ml.eval_holdout` on one
# card and the longer wall clock.

sh(
    sys.executable,
    "-m",
    "ml.eval_sharded",
    "--self-test",
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 8: the collective DDP depends on ------------------------------
#
# One all-reduce across both cards, under the same launcher the fine-tune
# uses. About a minute, most of it CUDA context creation.
#
# Worth its own cell because of how the failure it catches presents: a
# misconfigured NCCL does not raise, it *waits*. The training cell would
# print its banner, allocate both replicas, and then sit at step 0 until
# the session expires -- and from the outside that is indistinguishable
# from a slow first step. Here the same failure costs a minute and says
# so, twice over: the process group has a 120 s timeout of its own, and
# `sh` has an outer one.
#
# The value is checked, not just the absence of an exception. The ranks
# contribute 1 and 2, so a correct sum is 3 in every element. A
# collective that completes with the wrong contents is a wrong gradient,
# which is worse than a hang, because it produces an adapter.

preflight = Path(tempfile.gettempdir()) / "drivemind_nccl_preflight.py"

preflight.write_text(
    '''
import datetime
import os

import torch
import torch.distributed as dist

dist.init_process_group("nccl", timeout=datetime.timedelta(seconds=120))

rank = dist.get_rank()
world = dist.get_world_size()
local = int(os.environ["LOCAL_RANK"])

torch.cuda.set_device(local)

tensor = torch.full((1024, 1024), float(rank + 1), device="cuda")

dist.all_reduce(tensor)

expected = float(sum(range(1, world + 1)))
correct = bool((tensor == expected).all().item())

print(
    f"  rank {rank}/{world} on cuda:{local}"
    f" ({torch.cuda.get_device_name(local)}):"
    f" all_reduce -> {tensor[0, 0].item():.0f}, expected {expected:.0f}",
    flush=True,
)

dist.barrier()
dist.destroy_process_group()

raise SystemExit(0 if correct else 1)
''',
    encoding="utf-8",
)

if WORLD_SIZE < 2:
    print("  one card -- no collective to check, skipping")

else:
    torchrun(preflight, module=False, cwd=REPO, env=RUN_ENV, timeout=300)
    print("\n  Both cards can all-reduce, with the right answer.")

In [ ]:
# --- Cell 9: the training data path, tokenizer only ----------------------
#
# Encodes all 9,000 rows and stops. No GPU, no base weights -- just the
# tokenizer, which is a few megabytes.
#
# This is the step that catches a stale policy stamp and a row over the
# length budget. Both are refusals rather than warnings, and both would
# otherwise surface after the 4-bit weights are resident: on a
# session-limited runtime, finding out now is worth the two minutes.
#
# Read the reported token distribution. `max_length` is 1,024 and the
# encoder raises rather than truncating, because the tail of a long
# prompt is where `signals` lives -- the part of the evidence the label
# most depends on. A truncated example trains the model to answer a
# question it was not shown.

sh(
    sys.executable,
    "-m",
    "ml.train_qlora",
    "--model",
    MODEL,
    "--dry-run",
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 10: does the model load and emit the contract? ----------------
#
# 12 hand-built cases, about three minutes once the weights are cached.
# Not a measurement: a model can pass all twelve by pattern matching on
# `category`, and the runner prints that caveat itself.
#
# It answers the one question worth answering before committing an hour --
# does this model load in 4-bit on a T4, and does it emit something
# `parse_ai_response` accepts.
#
# One card, and it has to be this cell rather than a later one. The first
# run downloads about 8 GB of base weights, and every GPU cell after this
# one starts two processes that would both want them. Two Hugging Face
# downloads into one cache directory is not a race worth having on a
# metered session, so the download happens once, in a single process,
# here.

sh(
    sys.executable,
    "-m",
    "ml.eval_holdout",
    "--model",
    MODEL,
    "--smoke",
    cwd=REPO,
    env=ONE_CARD_ENV,
)

In [ ]:
# --- Cell 11: twenty steps on the real weights, both cards --------------
#
# The fine-tune, under the real launcher, on every card, stopped after 20
# optimizer steps. About ten minutes, and it buys three things the dry run
# cannot:
#
#   * the per-rank batch actually fits. PER_DEVICE_BATCH is a guess until
#     a T4 has held a 4-bit replica, the gradient-checkpointed
#     activations, the LoRA gradients and a paged optimizer state at once.
#     An OOM here costs ten minutes; the same OOM ninety minutes into the
#     real run costs the run.
#   * a throughput number. `--eval-steps 20` puts one validation pass and
#     one checkpoint save inside those 20 steps, so the whole loop is
#     exercised rather than just the forward path, and the rate in the
#     progress bar turns the "1-2 h" estimate above into arithmetic.
#   * the adapter write path. A run that trains for two hours and then
#     fails in `save_pretrained` has produced nothing.
#
# It writes to `artifacts/qlora_calibration`, not `artifacts/qlora`, and
# `train_qlora` stamps `partial_run_max_steps` into its
# `drivemind_run.json`. Twice over, a 20-step adapter cannot be mistaken
# for the finished one.

torchrun(
    "ml.train_qlora",
    "--model",
    MODEL,
    "--out",
    ARTIFACTS / "qlora_calibration",
    "--batch-size",
    PER_DEVICE_BATCH,
    "--grad-accum",
    GRAD_ACCUM,
    "--max-steps",
    20,
    "--eval-steps",
    20,
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 12: the base model, held out, across every card ---------------
#
# 1,126 cases -- 1,000 test, 100 gold, 26 red_team -- greedy. Estimated 25
# minutes on two cards, unmeasured.
#
# This has to happen *before* training, and in this environment. A base
# figure taken from a different runtime is not a comparison, and a base
# figure taken after the adapter exists is a figure nobody will trust.
#
# What is parallel and what is not: one worker per card generates an
# interleaved share of the cases -- interleaved rather than a first half
# and a second half, because `red_team` sits entirely at the end of the
# enumeration and a contiguous split would hand one card the whole
# adversarial tail while the other waited. Nothing is scored until every
# worker has finished, and then all 1,126 are scored by a single
# `evaluate_emitter` call in the parent, which regenerates each case and
# verifies it against the row on disk exactly as the single-card path
# does. There is no --limit.
#
# If a worker fails, or any case is generated twice or not at all, the run
# refuses and prints no report. Cell 7 exercised those refusals on the
# CPU. The per-shard raw generations are kept next to the report, in
# `artifacts/_shards_base_holdout/`.

sh(
    sys.executable,
    "-m",
    "ml.eval_sharded",
    "--model",
    MODEL,
    "--devices",
    DEVICES,
    "--out",
    str(ARTIFACTS / "base_holdout.txt"),
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 13: QLoRA fine-tune, one rank per card ------------------------
#
# 8,000 rows at an effective batch of 16 is 500 optimizer steps for one
# epoch, and each step is split across the ranks. Estimated 1-2 hours on
# two cards, unmeasured.
#
# Data parallel, not model parallel: each rank holds a complete 4-bit
# replica and a complete copy of the LoRA parameters, sees a disjoint half
# of every batch, and the gradients are all-reduced before the step. The
# adapter that comes out is the same fine-tune a single-card run of this
# recipe would produce in intent -- same effective batch, same learning
# rate, same schedule -- but it is not bit-for-bit that adapter: the data
# order and the gradient reduction both differ. `drivemind_run.json`
# records `world_size` and the batch decomposition for exactly that
# reason, because a figure from a two-rank run and a figure from a
# one-rank run are two measurements, not one repeated.
#
# `--eval-steps 125` rather than the default 50, and the reason is
# arithmetic: a validation pass is 1,000 rows against 8,000 for the whole
# epoch of training. At every 50 steps that is ten passes -- more forward
# passes spent on checkpoint selection than on the epoch itself. Four
# passes still gives `load_best_model_at_end` something to choose between.
#
# Selection is on `eval_loss` over validation.jsonl, which is a proxy and
# is labelled as one: lower validation loss is not higher action
# agreement. Measuring agreement needs generation, and generating over
# 1,000 cases at every checkpoint costs more than the fine-tune. The
# metrics that matter are measured once, in the next cell.
#
# `test.jsonl` is not opened by this script. A checkpoint chosen by test
# performance has spent the test set.

torchrun(
    "ml.train_qlora",
    "--model",
    MODEL,
    "--out",
    ARTIFACTS / "qlora",
    "--batch-size",
    PER_DEVICE_BATCH,
    "--grad-accum",
    GRAD_ACCUM,
    "--eval-steps",
    125,
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 14: the fine-tuned model, held out ----------------------------
#
# Cell 12's command plus `--adapter`: same cards, same shard split, same
# corpus, same decoding, same scorer, same process structure. That is the
# point of running it this way -- the only thing that differs between the
# two reports is the adapter.

sh(
    sys.executable,
    "-m",
    "ml.eval_sharded",
    "--model",
    MODEL,
    "--adapter",
    str(ARTIFACTS / "qlora"),
    "--devices",
    DEVICES,
    "--out",
    str(ARTIFACTS / "tuned_holdout.txt"),
    cwd=REPO,
    env=RUN_ENV,
)

In [ ]:
# --- Cell 15: both reports, and how to read them -------------------------
#
# Printed together so the pair travels as one artifact. Everything in
# `artifacts/` under /kaggle/working is saved as notebook output: the two
# reports, the adapter, `drivemind_run.json` recording which policy's
# labels it learned and which base model it belongs on top of, and the
# per-shard raw generations behind each report. An adapter without that
# JSON says neither of those things.

for name in ("base_holdout.txt", "tuned_holdout.txt"):
    path = ARTIFACTS / name

    print("=" * 72)
    print(name)
    print("=" * 72)
    print(path.read_text(encoding="utf-8") if path.exists() else "  not written")
    print()

run_json = ARTIFACTS / "qlora" / "drivemind_run.json"

if run_json.exists():
    print("=" * 72)
    print("drivemind_run.json")
    print("=" * 72)
    print(run_json.read_text(encoding="utf-8"))

print("=" * 72)
print("artifacts")
print("=" * 72)

for path in sorted(ARTIFACTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACTS)!s:<44}{path.stat().st_size:>12,} B")

print()
print("  Read unsafe escalation rate first, not action agreement.")
print("  It is the primary safety metric and it is counted BEFORE the")
print("  gate clamps, so it measures what the model attempted rather")
print("  than what a user would have seen. A fine-tune that raises")
print("  agreement while attempting more escalations is worse, not")
print("  better, and reading agreement first would hide that.")
print()
print("  Structured-output validity is the second thing to read. A model")
print("  that cannot hold the {action, risk, explanation} contract has")
print("  no agreement figure worth discussing -- an unparseable response")
print("  lands the user on the deterministic recommendation, which is")
print("  safe and also means the model contributed nothing.")
print()
print("  Neither figure is affected by having run on two cards. The")
print("  generation was split; the scoring was one call over all 1,126")
print("  cases, and both reports name the placement they came from so")
print("  that a later single-card run can be compared against them.")
print()
print("  Both reports and drivemind_run.json are what to paste back.")